# DATA Hypothesis - VANGUARD A/B TEST
---

** Import Libraries**

In [141]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

sns.set_style("whitegrid")

**Loading Database**

In [142]:
data_path = "data/processed"
web = pd.read_csv(f"{data_path}/web.csv")

## Phase 2 KPIs 

**GLOBAL COMPLETION RATE**
- Number of clients who finished the process in correct order, meaning the client reached the confirm step without errors: clients who went through the whole process smoothly

**COMPLETION RATE**
- Number of correct: if user managed to go through the process with less repetitions and errors, but still reached the confirmation step

**ERROR RATES**
- If there a step where users go back to a previous step. Moving from a later step to an earlier one is considered an error.

**ERROR RATE BY REPETITION**
- If there is a step where users repeatedly go back to. Moving back and forth on the same step. This metric serves as an indicator of possible UI errors in specific steps

**DROP-OUT RATE**
- Number of clients that didn't manage to reach the last stage (confirm) and abandoned the process before

**TIME SPENT ON EACH STEP**
- The average duration users spend on each step.

GLOBAL COMPLETION RATE

In [143]:
# 1. Define the ideal sequence
ideal_path = ['start', 'step_1', 'step_2', 'step_3', 'confirm']

# 2. Logic to check each group
def check_journey(group):
    actual_steps = group['process_step'].tolist()
    actual_statuses = group['process_status'].tolist()
    
    # Check if sequence is exactly the ideal path AND all statuses are 'correct'
    return (actual_steps == ideal_path) and all(s == 'correct' for s in actual_statuses)

# 3. Calculate per client (This creates a Series with client_id as index)
global_success_map = web.groupby('client_id').apply(check_journey)

# 4. Map the result back to every row in the original dataframe
web['global_success'] = web['client_id'].map(global_success_map)

print(global_success_map)

client_id
555         True
647         True
934        False
1028       False
1104       False
           ...  
9999150    False
9999400     True
9999626    False
9999729    False
9999832    False
Length: 50500, dtype: bool


C:\Users\eymem\AppData\Local\Temp\ipykernel_12964\1463843925.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  global_success_map = web.groupby('client_id').apply(check_journey)


COMPLETION RATE (stored under variable is_completed)

In [144]:
# 1. Define the success logic for Completion Rate
def check_any_completion(group):
    """
    Checks if 'confirm' exists in the user's journey.
    """
    actual_steps = group['process_step'].tolist()
    
    # Standard Completion: Did they ever reach the end?
    return 'confirm' in actual_steps

# 2. Calculate per client
completion_map = web.groupby('client_id').apply(check_any_completion)

# 3. Map back to the dataframe to create the new column
web['is_completed'] = web['client_id'].map(completion_map)

# 4. Calculate the overall KPI percentage
# Since 'is_completed' is boolean, the mean() gives us the rate
completion_rate_total = web.drop_duplicates('client_id')['is_completed'].mean() * 100

print(f"Overall Completion Rate: {completion_rate_total:.22f}%")

Overall Completion Rate: 67.5663366336633686159985%


C:\Users\eymem\AppData\Local\Temp\ipykernel_12964\726274563.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  completion_map = web.groupby('client_id').apply(check_any_completion)


VARIATION OF COMPLETION RATE (INCL. POSSIBLE ERRORS)

In [145]:
# 2. Group by Variation and Client_id
# We include variation so we can group by it in the final step
completion_by_user = web.groupby(['variation', 'client_id']).apply(check_any_completion)

# 3. Calculate the mean per variation and multiply by 100 for percentage
# Since the result of apply is a boolean, mean() gives the success ratio
completion_rate_report = completion_by_user.groupby('variation').mean() * 100

print("--- Overall Completion Rate by Variation ---")
print(completion_rate_report.round(2).apply(lambda x: f"{x}%"))

--- Overall Completion Rate by Variation ---
variation
Control    65.59%
Test       69.29%
dtype: object


C:\Users\eymem\AppData\Local\Temp\ipykernel_12964\584329963.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  completion_by_user = web.groupby(['variation', 'client_id']).apply(check_any_completion)


ERROR RATES (BACKWARDS ERROR) & (REPETITON ERROR)

In [146]:
web["backwards_error"] = (web["process_status"].map({"correct": 1, "repeated": 0, "error": 0}).astype("Int64"))

In [147]:
web.head(50)

,client_id,visitor_id,visit_id,process_step,process_order,process_status,date_time,next_time,sec_spent,variation,global_success,is_completed,backwards_error
0,555,402506806_56087378777,637149525_38041617439_716659,start,step_1,correct,2017-04-15 12:57:56,2017-04-15 12:58:03,7.0,Test,True,True,1
1,555,402506806_56087378777,637149525_38041617439_716659,step_1,step_2,correct,2017-04-15 12:58:03,2017-04-15 12:58:35,32.0,Test,True,True,1
2,555,402506806_56087378777,637149525_38041617439_716659,step_2,step_3,correct,2017-04-15 12:58:35,2017-04-15 13:00:14,99.0,Test,True,True,1
3,555,402506806_56087378777,637149525_38041617439_716659,step_3,confirm,correct,2017-04-15 13:00:14,2017-04-15 13:00:34,20.0,Test,True,True,1
4,555,402506806_56087378777,637149525_38041617439_716659,confirm,NaN,correct,2017-04-15 13:00:34,NaN,NaN,Test,True,True,1
5,647,66758770_53988066587,40369564_40101682850_311847,start,step_1,correct,2017-04-12 15:41:28,2017-04-12 15:41:35,7.0,Test,True,True,1
6,647,66758770_53988066587,40369564_40101682850_311847,step_1,step_2,correct,2017-04-12 15:41:35,2017-04-12 15:41:53,18.0,Test,True,True,1
7,647,66758770_53988066587,40369564_40101682850_311847,step_2,step_3,correct,2017-04-12 15:41:53,2017-04-12 15:45:02,189.0,Test,True,True,1
8,647,66758770_53988066587,40369564_40101682850_311847,step_3,confirm,correct,2017-04-12 15:45:02,2017-04-12 15:47:45,163.0,Test,True,True,1
9,647,66758770_53988066587,40369564_40101682850_311847,confirm,NaN,correct,2017-04-12 15:47:45,NaN,NaN,Test,True,True,1


OVERALL DROP-OUT RATE

In [148]:
# 1. Identify the last step for each client
last_steps = web.sort_values(['client_id', 'date_time']).groupby('client_id')['process_step'].last()

# 2. A drop-out is anyone whose last step is NOT 'confirm'
is_dropout = last_steps != 'confirm'

# 3. Calculate Rate per variation
dropout_report = is_dropout.to_frame().join(web[['client_id', 'variation']].drop_duplicates().set_index('client_id'))
dropout_rate = dropout_report.groupby('variation')['process_step'].mean() * 100

print("--- Drop-Out Rate (%) ---")
print(dropout_rate.round(2))

--- Drop-Out Rate (%) ---
variation
Control    41.99
Test       32.71
Name: process_step, dtype: float64


DROP-OUT RATE BY STEP

In [149]:
# 1. Sort globally to ensure the last row is the chronologically final interaction
web = web.sort_values(['client_id', 'date_time'])

# 2. Get the total unique clients per variation (Denominator)
total_clients_per_group = web.groupby('variation')['client_id'].nunique()

# 3. Identify the final step for every client
# .tail(1) after groupby gives us the very last record for each client
final_steps_per_client = web.groupby('client_id').tail(1)

# 4. Count where users ended their journey
drop_off_counts = final_steps_per_client.groupby(['variation', 'process_step'])['client_id'].count().unstack(fill_value=0)

# 5. Calculate Drop-off Rate (%)
# This shows: "What % of total clients in this group stopped at this specific step?"
drop_off_rate_by_step = (drop_off_counts.div(total_clients_per_group, axis=0) * 100)

# 6. Reorder for logical funnel flow
step_order = ['start', 'step_1', 'step_2', 'step_3', 'confirm']
drop_off_rate_by_step = drop_off_rate_by_step[[col for col in step_order if col in drop_off_rate_by_step.columns]]

print("--- Drop-off Rate by Step (Final Interaction per Client) ---")
display(drop_off_rate_by_step.round(2))

--- Drop-off Rate by Step (Final Interaction per Client) ---


process_step,start,step_1,step_2,step_3,confirm
variation,,,,,
Control,23.72,9.46,3.12,5.69,58.01
Test,20.68,6.13,2.11,3.78,67.29


TIME SPENT ON EACH STEP

In [150]:
# 1. Ensure chronological order within each session
web['date_time'] = pd.to_datetime(web['date_time'])
web = web.sort_values(['client_id', 'visit_id', 'date_time'])

# 2. Group by BOTH client and visit to isolate sessions
# We use .diff() and .shift(-1) within the session group
web['seconds_spent'] = web.groupby(['client_id', 'visit_id'])['date_time'].diff().shift(-1).dt.total_seconds()

# 4. Summarize Average Time per Step and Variation
time_summary = web.groupby(['variation', 'process_step'])['seconds_spent'].mean()

print("--- Average Seconds Spent per Step (Per Session) ---")
display(time_summary.round(2))

--- Average Seconds Spent per Step (Per Session) ---


variation  process_step
Control    confirm         153.00
           start            62.91
           step_1           50.23
           step_2           91.58
           step_3          135.38
Test       confirm         236.24
           start            60.56
           step_1           60.50
           step_2           88.59
           step_3          128.95
Name: seconds_spent, dtype: float64

In [151]:

# 1. First, create the column in the main 'web' dataframe
# Make sure 'date_time' is converted to datetime objects first!
web['date_time'] = pd.to_datetime(web['date_time'])
web = web.sort_values(['client_id', 'visit_id', 'date_time'])

# This creates the 'seconds_spent' column that the function is looking for
web['seconds_spent'] = web.groupby(['client_id', 'visit_id'])['date_time'].diff().shift(-1).dt.total_seconds()

# 2. Define the rank mapping
step_ranks = {'start': 1, 'step_1': 2, 'step_2': 3, 'step_3': 4, 'confirm': 5}
web['step_rank'] = web['process_step'].map(step_ranks)

# 3. Now run the aggregation
def aggregate_client_kpis(group):
    completed = 'confirm' in group['process_step'].values
    
    # Force numeric and drop NaNs for the math
    ranks = pd.to_numeric(group['step_rank'], errors='coerce').dropna().tolist()
    
    backwards = False
    is_global = False
    
    if len(ranks) >= 2:
        diffs = [ranks[i] - ranks[i-1] for i in range(1, len(ranks))]
        backwards = any(d < 0 for d in diffs)
        is_global = (ranks == [1, 2, 3, 4, 5])
    
    # We use .get() or check existence to be extra safe
    avg_time = 0
    if 'seconds_spent' in group.columns:
        avg_time = group['seconds_spent'].mean()

    return pd.Series({
        'variation': group['variation'].iloc[0],
        'is_completed': int(completed),
        'is_global_success': int(is_global),
        'has_backwards_error': int(backwards),
        'avg_time_per_step': round(avg_time, 2) if pd.notnull(avg_time) else 0
    })

# 4. Apply
df_clients = web.groupby('client_id').apply(aggregate_client_kpis).reset_index()

C:\Users\eymem\AppData\Local\Temp\ipykernel_12964\2048394068.py:42: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_clients = web.groupby('client_id').apply(aggregate_client_kpis).reset_index()


In [152]:
df_clients.head(10)

,client_id,variation,is_completed,is_global_success,has_backwards_error,avg_time_per_step
0,555,Test,1,1,0,39.50
1,647,Test,1,1,0,94.25
2,934,Test,0,0,0,47.33
3,1028,Control,0,0,1,67.25
4,1104,Control,0,0,0,0.00
5,1186,Control,0,0,0,11.00
6,1195,Control,1,1,0,61.25
7,1197,Control,1,0,1,15.83
8,1336,Test,1,0,0,58.60
9,1346,Test,0,0,1,55.00


In [153]:
web.columns

Index(['client_id', 'visitor_id', 'visit_id', 'process_step', 'process_order',
       'process_status', 'date_time', 'next_time', 'sec_spent', 'variation',
       'global_success', 'is_completed', 'backwards_error', 'seconds_spent',
       'step_rank'],
      dtype='object')

In [154]:
# 1. Define the columns to keep
cols_to_keep = [
    'client_id', 'visitor_id', 'visit_id', 'process_step', 
    'process_status', 'date_time', 'variation',
    'is_completed', 'backwards_error',
    'seconds_spent'
]

# 2. Use .copy() to ensure we are working on a fresh DataFrame
# This prevents the SettingWithCopyWarning
web = web[cols_to_keep].copy()

# 3. Now rounding will work perfectly without warnings
web['seconds_spent'] = web['seconds_spent'].round(2)

In [155]:
web.head()

,client_id,visitor_id,visit_id,process_step,process_status,date_time,variation,is_completed,backwards_error,seconds_spent
0,555,402506806_56087378777,637149525_38041617439_716659,start,correct,2017-04-15 12:57:56,Test,True,1,7.0
1,555,402506806_56087378777,637149525_38041617439_716659,step_1,correct,2017-04-15 12:58:03,Test,True,1,32.0
2,555,402506806_56087378777,637149525_38041617439_716659,step_2,correct,2017-04-15 12:58:35,Test,True,1,99.0
3,555,402506806_56087378777,637149525_38041617439_716659,step_3,correct,2017-04-15 13:00:14,Test,True,1,20.0
4,555,402506806_56087378777,637149525_38041617439_716659,confirm,correct,2017-04-15 13:00:34,Test,True,1,NaN


In [163]:
df_clients.head()

,client_id,variation,is_completed,is_global_success,has_backwards_error,avg_time_per_step
0,555,Test,1,1,0,39.50
1,647,Test,1,1,0,94.25
2,934,Test,0,0,0,47.33
3,1028,Control,0,0,1,67.25
4,1104,Control,0,0,0,0.00


In [156]:
import os

# Define the output directory
output_dir = "data/processed"
os.makedirs(output_dir, exist_ok=True)

# Exporting both the transactional (web) and summary (clients) datasets
# Assuming your enriched dataframe is named 'web' and the summary is 'df_clients'
web.to_csv(f"{output_dir}/vanguard_web_wrangled.csv", index=False)
df_clients.to_csv(f"{output_dir}/vanguard_client_summary.csv", index=False)

print("Exported:")
for f in os.listdir(output_dir):
    path = os.path.join(output_dir, f)
    if os.path.isfile(path):
        size = os.path.getsize(path)
        print(f"  {f} ({size:,} bytes)")

Exported:
  df_exde.csv (3,302,143 bytes)
  df_web_3.csv (81,420,206 bytes)
  vanguard_client_summary.csv (1,354,845 bytes)
  vanguard_hypo.csv (3,884,978 bytes)
  vanguard_web_wrangled.csv (35,857,674 bytes)
  web.csv (40,471,437 bytes)
